# Session 2 — Qubits & States

**Author:** PPSP lab  **·**  **Date:** 2026-05-27  **·**  **Project:** Quantum Optimal Transport course  **·**  **Status:** 🔬 Teaching

## Purpose

We start the quantum side from zero. A *qubit* is the quantum version of a bit — but
instead of being 0 or 1, it lives on a sphere. We build it from amplitudes, see the
**Born rule** turn amplitudes into measurement probabilities, picture it on the **Bloch
sphere**, and watch **shot noise** on both a simulator and (optionally) real hardware.

## 0. What you'll be able to do

- Write a qubit as a 2-component complex vector and read its amplitudes.
- Apply the **Born rule**: amplitude → measurement probability.
- Locate a state on the **Bloch sphere**.
- Explain **shot noise** and why more shots converge to the Born probabilities.
- Build and run a one-qubit circuit in Qiskit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.intro import check_environment
from qot_course.quantum.states import (
    KET_0, KET_1, KET_PLUS, qubit_state, born_probabilities, sample_counts,
)

np.random.seed(42)
viz.use_course_style()
print("qiskit:", check_environment()["qiskit"])

## 1. From a bit to a qubit

A classical bit is 0 or 1. A **qubit** is a unit vector in a 2-dimensional complex space:

$$ |\psi\rangle = a_0\,|0\rangle + a_1\,|1\rangle, \qquad |a_0|^2 + |a_1|^2 = 1. $$

The numbers $a_0, a_1$ are **complex amplitudes**. The special state
$|+\rangle = \tfrac{1}{\sqrt 2}(|0\rangle + |1\rangle)$ is an equal **superposition**.

In [ ]:
print("|0> =", np.round(KET_0, 3))
print("|1> =", np.round(KET_1, 3))
print("|+> =", np.round(KET_PLUS, 3))

# a general state on the sphere, parameterised by two angles:
psi = qubit_state(theta=2 * np.pi / 3, phi=np.pi / 4)
print("a tilted state |psi> =", np.round(psi, 3))
print("norm:", round(float(np.linalg.norm(psi)), 6))

**Read the output.** `|0>` and `|1>` are the "classical" basis states. `|+>` puts equal
amplitude on both. The tilted `|psi>` has a *complex* amplitude on `|1>` (a phase) — yet
its norm is 1, as every valid qubit must be.

## 2. The Born rule

You cannot read the amplitudes directly. When you **measure** in the computational basis,
you get outcome $x$ with probability $|a_x|^2$ — the **Born rule**. Measurement is
intrinsically random.

In [ ]:
for name, state in [("|0>", KET_0), ("|+>", KET_PLUS), ("|psi>", psi)]:
    print(f"{name}: P =", {k: round(v, 3) for k, v in born_probabilities(state).items()})

**Read the output.** `|0>` always gives 0. `|+>` is a perfect coin flip (0.5 / 0.5).
The tilted state is biased — and notice the *phase* of `|psi>` did **not** change these
probabilities: a Z-measurement is blind to phase. (Holding onto that fact pays off later.)

## 3. The Bloch sphere

Every pure qubit is a point on a sphere: the poles are $|0\rangle$ (up) and $|1\rangle$
(down), the equator holds the equal superpositions like $|+\rangle$. The angles
$(\theta, \phi)$ are exactly the ones we used in `qubit_state`.

In [ ]:
viz.plot_bloch(KET_PLUS, title="|+>  (equator)")
viz.plot_bloch(psi, title="|psi>  (tilted)")
plt.show()

**Read the figures.** `|+>` sits on the equator (an equal superposition); the tilted
`|psi>` points up-and-around, its longitude set by the phase $\phi$ and its latitude by
$\theta$. The Bloch sphere is our compass for everything that follows.

## 4. Measurement & shot noise

One measurement gives a single 0 or 1. To estimate the Born probabilities we repeat the
measurement many times (**shots**) and count. With few shots the estimate is noisy; with
many it converges. Let's measure $|+\rangle$ at 20, 200 and 2000 shots.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
for ax, shots in zip(axes, [20, 200, 2000]):
    counts = sample_counts(KET_PLUS, shots=shots, seed=0)
    viz.plot_counts(counts, ax=ax)
    ax.set_title(f"{shots} shots", pad=10)
fig.suptitle("Measuring |+>: more shots converge toward the 50/50 Born probabilities",
             fontsize=15, fontweight="bold")
fig.tight_layout()
plt.show()

**Read the figure.** At 20 shots the split can be visibly lopsided (pure chance); by
2000 shots it is close to the true 50/50. This fluctuation is **shot noise** — it is not
a bug, it is what sampling a random process looks like.

## 5. A real circuit with Qiskit

The state $|+\rangle$ is made by applying a **Hadamard gate** `H` to $|0\rangle$. Let's
build that circuit, look at it, and run it on a simulator.

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

qc = QuantumCircuit(1, 1)
qc.h(0)            # |0> -> |+>
qc.measure(0, 0)   # measure in the computational basis
qc.draw("mpl")

In [ ]:
sim = AerSimulator()
result = sim.run(transpile(qc, sim), shots=2048, seed_simulator=42).result()
counts = result.get_counts()
print("counts:", counts)

viz.plot_counts(counts)
plt.show()

**Read the figure.** The simulator reproduces the ideal coin flip (≈ 1024 / 1024).

On **real hardware** the split is *not* exactly even — read-out and gate errors bias it.
That is the honest reason the next session needs **density matrices**: a prepared state
is never perfectly pure. The verified idiom to run this on an IBM QPU (Open Plan, free):

```python
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
service = QiskitRuntimeService()                         # uses your saved account (no token in code)
backend = service.least_busy(operational=True, simulator=False)
sampler = SamplerV2(mode=backend)
job = sampler.run([transpile(qc, backend)])
counts = job.result()[0].data.c.get_counts()
```

## 6. Dictionary update

| Classical | Quantum |
|-----------|---------|
| probability vector `p` | density matrix $\rho$ (diagonal $\Rightarrow$ classical) |
| marginal | partial trace |
| event probability $p(x)$ | **Born rule** $|\langle x|\psi\rangle|^2$ |

## 7. Your turn

1. **Rotate the state.** Pick your own `theta`, `phi` in `qubit_state`, predict the Bloch
   vector and the measurement counts, then check with `viz.plot_bloch` and `sample_counts`.
2. **Phase is invisible to Z.** Change only `phi` and confirm `born_probabilities` does not
   move. Why? (We'll see in S3 what *does* see the phase.)
3. **Build $|->$.** Apply `H` then `Z` to $|0\rangle$ in a circuit and measure. What do you get?

## Conclusions & references

- A qubit = a unit complex 2-vector = a point on the **Bloch sphere**.
- The **Born rule** turns amplitudes into measurement probabilities; sampling them gives shot noise.
- Real hardware is noisy → prepared states are *mixed* → we need density matrices (**S3**).

**References:** Nielsen & Chuang, *Quantum Computation and Quantum Information*, ch. 1–2; Qiskit textbook, "Representing Qubit States". Previous session: `notebooks/00_GettingStarted/01_roadmap.ipynb`.